# Neuron Migration - t7 (tax audit) without target values

For the surprise task t7 we had no target values at all. The same method still works, with three other sources of
evidence instead of targets:

1. **Weak labels**: Neuron's tax bookings (NRTAXB) say *which* tax was booked for which payout and year, only the
   amounts are missing. A candidate rule must book exactly the same taxes (`tools/weak_labels.py`).
2. **The two examples of the task text**: a convention search over the open questions must reproduce them to the cent
   (`tools/convention_search.py`).
3. **Consistency**: every tax Neuron booked must get an amount.

# Import libraries

In [1]:
import sys
import datetime as dt
import pandas as pd
from decimal import Decimal, ROUND_DOWN, ROUND_HALF_UP
from dateutil.relativedelta import relativedelta

sys.path.insert(0, '..')
from rules import load, CENT
from taxes import TaxModule, tax_results
from stamp_duty import stamp_duty
from tools.weak_labels import agreement, share_agreeing, explain_disagreement
from tools.convention_search import search

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)

# Hyperparameters

In [2]:
legacy_dir = '../../../legacy'

data = load(legacy_dir)

module = TaxModule(data)

# 1 Weak labels: which taxes did Neuron book?

Every payout (withdrawal, surrender, death benefit) with the residence of its policyholder and the taxes Neuron booked
for it.

In [3]:
cases = []
for policy, payouts in module.payouts().items():
    withdrawals = []
    for payout in payouts:
        event, day, _, amount, share = payout
        address = module.residence.address(policy, day)
        country = address.AD_CNTRY
        terms = module.policies[policy]
        cases.append({
            'policy': policy, 'event': event, 'day': day, 'country': country, 'objection': address.AD_TAXST == 'E',
            'church': address.AD_CHURCH == 'J',
            'payout_chf': float(module.convert(amount, policy, 'CH', day)),
            'gain': float(module.gain(policy, payout, withdrawals, country)) if country in ('DE', 'IT') else None,
            'years_held': relativedelta(day, terms.PH_BEGDT.date()).years, 'applied': terms.PH_APPDT,
            'taxes': '+'.join(sorted(module.booked[(policy, event)])) or '-'})
        withdrawals.append((day, share))
cases = pd.DataFrame(cases)
events = data.table('NREVNT').set_index(['EV_POLNR', 'EV_EVTNR']).EV_EVTTP
cases['death'] = [events[(p, e)] == 500 for p, e in zip(cases.policy, cases.event)]
pd.crosstab(cases.country, cases.taxes)

taxes,-,BOC,BOC+BOL,BOC+BOL+SOS,BOC+KIS+KST+SOL,BOC+KST+SOL,BOC+SOS,BOC+VST,KIS+KST+SOL,KST+SOL,SOS,VST
country,,,,,,,,,,,,
CH,1397,6,0,0,0,0,0,2,0,0,0,255
DE,195,1,1,0,1,1,0,0,28,58,0,0
IT,66,90,83,50,0,0,59,0,0,0,44,0
LU,153,1,0,0,0,0,0,0,0,0,0,0


Each country has its own taxes: VST only in Switzerland, SOS in Italy, KST/SOL/KIS in Germany, nothing in
Luxembourg. The Italian stamp duty (BOC) is also collected from holders who moved away from Italy.

## Swiss withholding tax: objection to reporting and a threshold

In [4]:
swiss = cases[cases.country == 'CH']
rule = swiss.objection & (swiss.payout_chf > 5000)
booked = swiss.taxes.str.contains('VST')
print('agreement:', round(share_agreeing(rule, booked), 4))
agreement(rule, booked)

agreement: 0.9988


booked,False,True
rule says yes,,
False,1401,0
True,2,257


VST is withheld when the holder objected to the reporting (status E) and the payout is above CHF 5,000.

## Taxes on the gain: Italy and Germany

In [5]:
for country, tax in [('IT', 'SOS'), ('DE', 'KST')]:
    part = cases[(cases.country == country) & cases.gain.notna()]
    print(country, tax, 'agreement of \"gain > 0\" with the bookings:', round(share_agreeing(part.gain > 0, part.taxes.str.contains(tax)), 3))

IT SOS agreement of "gain > 0" with the bookings: 0.977
DE KST agreement of "gain > 0" with the bookings: 0.895


In [6]:
german = cases[(cases.country == 'DE') & cases.gain.notna()].copy()
german['APPLIED'] = german.applied
german['YEARS_HELD'] = german.years_held
german['DEATH'] = german.death
rules = explain_disagreement(german[['APPLIED', 'YEARS_HELD', 'DEATH']], german.gain > 0, german.taxes.str.contains('KST'))

30 of 285 values wrong (10.53%)
|--- APPLIED <= 2004.85
|   |--- YEARS_HELD <= 11.50
|   |   |--- APPLIED <= 1996.79
|   |   |   |--- class: wrong
|   |   |--- APPLIED >  1996.79
|   |   |   |--- class: right
|   |--- YEARS_HELD >  11.50
|   |   |--- APPLIED <= 1994.89
|   |   |   |--- class: wrong
|   |   |--- APPLIED >  1994.89
|   |   |   |--- class: wrong
|--- APPLIED >  2004.85
|   |--- YEARS_HELD <= 0.50
|   |   |--- APPLIED <= 2017.83
|   |   |   |--- class: right
|   |   |--- APPLIED >  2017.83
|   |   |   |--- class: wrong
|   |--- YEARS_HELD >  0.50
|   |   |--- DEATH <= 0.50
|   |   |   |--- class: right
|   |   |--- DEATH >  0.50
|   |   |   |--- class: wrong



Where a German gain was not taxed, the tree points to contracts applied for before 2005 and paid out after 12 years or
more (the old German rule for life insurance) and to death benefits. The solution does not need to re-implement these
exemptions: it takes *which* taxes apply from Neuron's bookings and calculates their amounts.

# 2 The examples of the task text

## German taxable base: BL10000086, surrender of 2006-07-17 (5,628.16)
Open questions: on which day are the premiums converted into EUR, and how are the converted amounts rounded?

In [7]:
policy, event = 'BL10000086', 121
payout = next(p for p in module.payouts()[policy] if p[0] == event)
premiums = module.premiums[policy]
paid = data.table('NRPREM').query('PR_POLNR == @policy and PR_STATE == "P"').set_index('PR_EVTNR').PR_DUEDT
rate = lambda day: module.market.rate('USD', day) / module.market.rate('EUR', day)

def german_base(choice):
    mode = {'cut off': ROUND_DOWN, 'half up': ROUND_HALF_UP}[choice['rounding']]
    day_of = {'money received': lambda day, number: day, 'due date': lambda day, number: paid[number].date()}[choice['premium day']]
    basis = sum((amount * rate(day_of(day, number))).quantize(CENT, mode) for day, amount, number in premiums if day <= payout[1])
    return int((payout[3] * rate(payout[1])).quantize(CENT, mode) - basis == Decimal('5628.16'))

search({'premium day': ['money received', 'due date'], 'rounding': ['cut off', 'half up']}, german_base)

,premium day,rounding,exact
0,money received,cut off,1
1,money received,half up,0
2,due date,cut off,0
3,due date,half up,0


Only premiums converted on the day the money was received, cut off to the cent (COBOL era), give 5,628.16.

## Italian stamp duty: BL10000044, 2019 (616.11)
Open questions: which exchange rate converts the policy value, and how is the rate correction of July 2019 booked?

In [8]:
policy = 'BL10000044'
values = {year: module.policy_value(policy, dt.date(year, 12, 31)) for year in [2017, 2018, 2019]}
to_eur = lambda value, day: value * module.market.rate('CHF', day) / module.market.rate('EUR', day)
fix_day = dt.date(2019, 7, 20)

def stamp_duty_2019(choice):
    day_of = (lambda year: dt.date(year, 12, 31)) if choice['conversion'] == '31 December' else (lambda year: fix_day)
    duty = lambda year, rate: (to_eur(values[year], day_of(year)) * rate).quantize(CENT, ROUND_HALF_UP)
    if choice['correction'] == 'new booking - old booking':
        fix = sum(duty(year, Decimal('0.002')) - duty(year, Decimal('0.0015')) for year in [2017, 2018])
    else:
        fix = sum(duty(year, Decimal('0.0005')) for year in [2017, 2018])
    return int(duty(2019, Decimal('0.002')) + fix == Decimal('616.11'))

search({'conversion': ['31 December', 'booking day'], 'correction': ['new booking - old booking', 'rate difference']},
       stamp_duty_2019)

,conversion,correction,exact
0,31 December,new booking - old booking,1
1,31 December,rate difference,0
2,booking day,new booking - old booking,0
3,booking day,rate difference,0


The policy value of 31 December converted with the rate of that day, and the correction booked as the new booking
minus the original one (both rounded): Neuron re-booked the years 2017 and 2018 when the rate was corrected in 2019.

# 3 The solution on the examples and the consistency check

In [9]:
payout_rows, annual_rows = tax_results(data)
print('BL10000086, 121:', payout_rows[('BL10000086', 121)])
print('BL10000044, 2019:', annual_rows[('BL10000044', 2019)])

BL10000086, 121: ('DE', Decimal('5628.16'), Decimal('1407.04'), Decimal('77.38'), Decimal('0.00'), datetime.date(2006, 8, 10))
BL10000044, 2019: (Decimal('0.00'), Decimal('0.00'), Decimal('0.00'), Decimal('616.11'), Decimal('0.00'), Decimal('0.00'), Decimal('0.00'))


In [10]:
withheld = {'VST': 2, 'SOS': 2, 'KST': 2, 'BOC': 4}
bookings = data.table('NRTAXB')
check = []
for row in bookings[bookings.TB_TAXTP.isin(list(withheld))].itertuples():
    result = payout_rows.get((row.TB_POLNR, row.TB_EVTNR))
    if result is not None:
        check.append({'tax': row.TB_TAXTP, 'amount calculated': result[withheld[row.TB_TAXTP]] > 0})
check = pd.DataFrame(check)
pd.crosstab(check.tax, check['amount calculated'])

amount calculated,True
tax,
BOC,295
KST,88
SOS,153
VST,257


Every tax that Neuron booked for a payout gets an amount.

# What is not confirmed yet

Without targets, these parts follow the law and the data structure but could not be checked to the cent:
- the remittance dates of Switzerland (30 days after the end of the month) and Italy (16th of the next month); the
  German one (10th of the next month) matches the example;
- the share of the year in the stamp duty booked at a payout (days / 365);
- the German partial exemption (from 2018) and half-income rule, which apply only to recent payouts;
- the taxable base of payouts without a withheld tax (0.00).

As soon as the organisers' actuary returns values for a few policies, the same tools (residuals, convention search)
will settle these points.